In [ ]:
%pip install -q transformers scikit-learn seaborn matplotlib pillow tqdm

from pathlib import Path
import json
import pandas as pd
import numpy as np

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPTokenizerFast, CLIPImageProcessor
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())


In [ ]:
ROOT = Path("/teamspace/studios/this_studio/Fakeddit")

TEST_CSV = ROOT / "subsets/test_10k_cached.csv"
TEST_IMG_DIR = ROOT / "images/test"

CKPT_PATH = ROOT / "outputs/cmcn_no_ocr_clip_l14/best_cmcn.pt"
OUT_DIR = ROOT / "outputs/cmcn_no_ocr_clip_l14/test_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "openai/clip-vit-large-patch14"
NUM_CLASSES = 6
MAX_LEN = 77
BATCH_SIZE = 64
NUM_WORKERS = 32
LABEL_COL = "6_way_label"

# This bias was selected from validation, not from test.
MISLEADING_BIAS = -0.25

CLASS_NAMES = [
    "True",
    "Satire",
    "Misleading Content",
    "Imposter Content",
    "False Connection",
    "Manipulated Content",
]

print("TEST_CSV:", TEST_CSV, TEST_CSV.exists())
print("TEST_IMG_DIR:", TEST_IMG_DIR, TEST_IMG_DIR.exists())
print("CKPT_PATH:", CKPT_PATH, CKPT_PATH.exists())
print("OUT_DIR:", OUT_DIR)


In [ ]:
test_df = pd.read_csv(TEST_CSV)

def resolve_text_col(df):
    for col in ["clean_title", "title", "clean_text", "text", "body"]:
        if col in df.columns:
            return col
    raise ValueError(f"No text column found. Columns: {list(df.columns)}")

TEXT_COL = resolve_text_col(test_df)

test_df["submission_id"] = test_df["submission_id"].astype(str)
test_df["image_path"] = test_df["submission_id"].apply(
    lambda x: str(TEST_IMG_DIR / f"{x}.jpg")
)
test_df["image_exists"] = test_df["image_path"].apply(lambda p: Path(p).exists())

print("test_df:", test_df.shape)
print("TEXT_COL:", TEXT_COL)
print("\nLabel distribution:")
print(test_df[LABEL_COL].value_counts().sort_index())
print("\nImage exists:")
print(test_df["image_exists"].value_counts())

assert test_df["image_exists"].all(), "Some test images are missing."
assert set(test_df[LABEL_COL].unique()).issubset(set(range(NUM_CLASSES)))

display(test_df[["submission_id", TEXT_COL, LABEL_COL, "image_path"]].head())


In [ ]:
print("Loading CLIP preprocessing...")
tokenizer = CLIPTokenizerFast.from_pretrained(MODEL_NAME)
image_processor = CLIPImageProcessor.from_pretrained(MODEL_NAME)
print("CLIP preprocessing ready")


In [ ]:
class FakedditTestDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row.get(TEXT_COL, "") or "").strip()
        if not text:
            text = "[empty]"

        enc = tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        try:
            image = Image.open(row["image_path"]).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), color=(0, 0, 0))

        pixels = image_processor(images=image, return_tensors="pt")["pixel_values"][0]

        return {
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
            "pixel_values": pixels,
            "labels": torch.tensor(int(row[LABEL_COL]), dtype=torch.long),
            "submission_id": row["submission_id"],
            "text": text,
            "image_path": row["image_path"],
        }

def collate(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
        "submission_id": [x["submission_id"] for x in batch],
        "text": [x["text"] for x in batch],
        "image_path": [x["image_path"] for x in batch],
    }

test_loader = DataLoader(
    FakedditTestDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate,
    persistent_workers=NUM_WORKERS > 0,
)

print("Test batches:", len(test_loader))


In [ ]:
class CMCN(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_classes=NUM_CLASSES):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(model_name)

        self.embed_dim = self.clip.config.projection_dim
        self.text_hidden_dim = self.clip.config.text_config.hidden_size
        self.vision_hidden_dim = self.clip.config.vision_config.hidden_size

        self.text_seq_proj = nn.Linear(self.text_hidden_dim, self.embed_dim)
        self.image_seq_proj = nn.Linear(self.vision_hidden_dim, self.embed_dim)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=8,
            dropout=0.10,
            batch_first=True,
        )
        self.cross_norm = nn.LayerNorm(self.embed_dim)

        self.text_head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Linear(self.embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_classes),
        )

        self.image_head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Linear(self.embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_classes),
        )

        self.gate = nn.Sequential(
            nn.Linear(2, 32),
            nn.GELU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

        final_dim = self.embed_dim * 3 + 2

        self.classifier = nn.Sequential(
            nn.Linear(final_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.30),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.20),

            nn.Linear(256, num_classes),
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        out = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True,
        )

        f_text = F.normalize(out.text_embeds, dim=-1)
        f_image = F.normalize(out.image_embeds, dim=-1)

        sim_score = F.cosine_similarity(f_text, f_image, dim=-1).unsqueeze(1)

        text_logits = self.text_head(f_text)
        image_logits = self.image_head(f_image)

        p_text = F.softmax(text_logits, dim=-1)
        p_image = F.softmax(image_logits, dim=-1)

        ambiguity = 0.5 * (
            F.kl_div(
                F.log_softmax(text_logits, dim=-1),
                p_image,
                reduction="none",
            ).sum(dim=1, keepdim=True)
            +
            F.kl_div(
                F.log_softmax(image_logits, dim=-1),
                p_text,
                reduction="none",
            ).sum(dim=1, keepdim=True)
        )

        text_tokens = self.text_seq_proj(out.text_model_output.last_hidden_state)
        image_tokens = self.image_seq_proj(out.vision_model_output.last_hidden_state[:, 1:, :])

        cross_tokens, _ = self.cross_attn(
            query=text_tokens,
            key=image_tokens,
            value=image_tokens,
            need_weights=False,
        )

        mask = attention_mask.unsqueeze(-1).to(cross_tokens.dtype)
        f_cross = (cross_tokens * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        f_cross = self.cross_norm(f_cross)

        alpha = self.gate(torch.cat([sim_score, ambiguity], dim=1))
        f_gated = alpha * f_text + (1.0 - alpha) * f_image

        f_final = torch.cat(
            [f_gated, f_cross, f_text, sim_score, ambiguity],
            dim=1,
        )

        logits = self.classifier(f_final)

        return {
            "logits": logits,
            "sim_score": sim_score.squeeze(1),
            "ambiguity": ambiguity.squeeze(1),
            "alpha": alpha.squeeze(1),
        }

print("CMCN class ready")


In [ ]:
def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

ckpt = load_checkpoint(CKPT_PATH)

print("Checkpoint epoch:", ckpt.get("epoch"))
print("Checkpoint stage:", ckpt.get("stage"))
print("Validation metrics:", ckpt.get("metrics"))
print("Checkpoint model:", ckpt.get("model_name"))

model = CMCN(MODEL_NAME, NUM_CLASSES).to(device)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.eval()

print("Model loaded")
print("Final feature dim:", model.embed_dim * 3 + 2)


In [ ]:
def amp_context():
    return torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=(device.type == "cuda" and torch.cuda.is_bf16_supported()),
    )

@torch.no_grad()
def evaluate_test(prefix="test", misleading_bias=None):
    model.eval()

    all_true = []
    all_pred = []
    rows = []

    for batch in tqdm(test_loader, desc=prefix):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with amp_context():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
            )

        logits = outputs["logits"].float()

        if misleading_bias is not None:
            logits[:, 2] += misleading_bias

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        labels_cpu = labels.detach().cpu().numpy()
        preds_cpu = preds.detach().cpu().numpy()
        probs_cpu = probs.detach().cpu().numpy()

        all_true.extend(labels_cpu.tolist())
        all_pred.extend(preds_cpu.tolist())

        sim_cpu = outputs["sim_score"].detach().float().cpu().numpy()
        amb_cpu = outputs["ambiguity"].detach().float().cpu().numpy()
        alpha_cpu = outputs["alpha"].detach().float().cpu().numpy()

        for i in range(len(preds_cpu)):
            row = {
                "submission_id": batch["submission_id"][i],
                "text": batch["text"][i],
                "image_path": batch["image_path"][i],
                "true_label_id": int(labels_cpu[i]),
                "true_label": CLASS_NAMES[int(labels_cpu[i])],
                "pred_label_id": int(preds_cpu[i]),
                "pred_label": CLASS_NAMES[int(preds_cpu[i])],
                "misleading_logit_bias": misleading_bias,
                "sim_score": float(sim_cpu[i]),
                "ambiguity": float(amb_cpu[i]),
                "gate_alpha": float(alpha_cpu[i]),
            }

            for j, name in enumerate(CLASS_NAMES):
                row[f"prob_{j}_{name}"] = float(probs_cpu[i, j])

            rows.append(row)

    pred_df = pd.DataFrame(rows)

    metrics = {
        "accuracy": accuracy_score(all_true, all_pred),
        "macro_f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(all_true, all_pred, average="weighted", zero_division=0),
        "misleading_logit_bias": misleading_bias,
    }

    report = classification_report(
        all_true,
        all_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        zero_division=0,
    )

    cm = confusion_matrix(
        all_true,
        all_pred,
        labels=list(range(NUM_CLASSES)),
    )

    with open(OUT_DIR / f"{prefix}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    with open(OUT_DIR / f"{prefix}_classification_report.txt", "w") as f:
        f.write(report)

    pred_df.to_csv(OUT_DIR / f"{prefix}_predictions.csv", index=False)

    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(prefix)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{prefix}_confusion_matrix.png", dpi=200)
    plt.show()

    print(prefix)
    print(json.dumps(metrics, indent=2))
    print(report)
    print("Saved to:", OUT_DIR)

    return metrics, pred_df

print("Evaluation function ready")


In [ ]:
raw_metrics, raw_pred_df = evaluate_test(
    prefix="test_raw",
    misleading_bias=None,
)


In [ ]:
cal_metrics, cal_pred_df = evaluate_test(
    prefix="test_calibrated_bias_m025",
    misleading_bias=MISLEADING_BIAS,
)

print("Final calibrated test metrics:")
print(json.dumps(cal_metrics, indent=2))


##with -.30 caliberation


In [ ]:
test_m030_metrics, test_m030_pred_df = evaluate_test(
    prefix="test_calibrated_bias_m030",
    misleading_bias=-0.30,
)

print("Final test metrics with bias -0.30:")
print(json.dumps(test_m030_metrics, indent=2))


In [ ]:
summary = pd.DataFrame([
    {"setting": "raw", **raw_metrics},
    {"setting": "bias_-0.25", **cal_metrics},
    {"setting": "bias_-0.30", **test_m030_metrics},
])

display(summary)
